# Treinar a voz do Líbero (RVC, grátis)

Treina um modelo da sua voz com os vídeos da pasta **Voz Líbero** do seu Google Drive (só a sua voz, sem música).
Leva uns 60–90 minutos numa GPU grátis do Colab. No fim baixa um `libero-rvc.zip`: mande esse arquivo de volta para o Claude (Drive ou upload).

**Como usar**
1. Menu **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4 → Salvar**.
2. Menu **Ambiente de execução → Executar tudo**.
3. No passo 3, autorize o acesso ao Google Drive (janela do Google → escolher a conta → **Permitir**).
4. Deixe a aba aberta até o último passo baixar o `libero-rvc.zip` (também é salvo no seu Drive, na pasta `Voz Líbero`).

Ferramenta: [Applio](https://github.com/IAHispano/Applio) (licença MIT). Use só com a sua própria voz.


In [ ]:
# 1. Confere a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "SEM GPU: troque o tipo de ambiente para GPU T4"


In [ ]:
# 2. Instala o Applio (~5 min)
%cd /content
!git clone --depth 1 https://github.com/IAHispano/Applio
%cd /content/Applio
!pip install -q -r requirements.txt
!python core.py prerequisites --pretraineds-hifigan --models --no-exe


In [ ]:
# 3. Pega o áudio dos vídeos da pasta "Voz Líbero" do seu Drive
import os, glob, subprocess
from google.colab import drive
drive.mount('/content/drive')
pasta = '/content/drive/MyDrive/Voz Líbero'
fontes = [f for f in glob.glob(pasta + '/*') if f.lower().endswith(('.mp4', '.mov', '.mkv', '.m4a', '.mp3', '.wav'))
          and 'libero-rvc' not in f]
assert fontes, 'Nenhum vídeo/áudio em "Voz Líbero" no Drive'
os.makedirs('/content/dataset', exist_ok=True)
for i, f in enumerate(fontes):
    # pula 8 s do começo (vinheta), tira grave de ruído e iguala o volume
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-ss', '8', '-i', f, '-vn', '-ac', '1', '-ar', '40000',
                    '-af', 'highpass=f=70,loudnorm=I=-18:TP=-2', f'/content/dataset/voz{i:02d}.wav'], check=True)
    print('ok', os.path.basename(f))
print(len(fontes), 'arquivos prontos para o treino')


In [ ]:
# 4. Prepara os dados
%cd /content/Applio
!python core.py preprocess --model-name libero --dataset-path /content/dataset --sample-rate 40000 --cpu-cores 2 --cut-preprocess Automatic
!python core.py extract --model-name libero --f0-method rmvpe --gpu 0 --cpu-cores 2 --sample-rate 40000


In [ ]:
# 5. Treina (~60–90 min). Não feche a aba.
%cd /content/Applio
!python core.py train --model-name libero --save-every-epoch 20 --save-only-latest --total-epoch 120 --sample-rate 40000 --batch-size 8 --gpu 0 --pretrained --index-algorithm Auto
!python core.py index --model-name libero --index-algorithm Auto


In [ ]:
# 6. Empacota o modelo, salva no Drive e baixa
import glob, os, zipfile
pasta = '/content/Applio/logs/libero'
pths = [p for p in glob.glob(pasta + '/*.pth') + glob.glob('/content/Applio/logs/*.pth')
        if not os.path.basename(p).startswith(('G_', 'D_'))]
idx = glob.glob(pasta + '/*.index')
pths.sort(key=os.path.getmtime)
assert pths, 'nenhum modelo .pth encontrado: veja se o passo 5 terminou sem erro'
with zipfile.ZipFile('/content/libero-rvc.zip', 'w') as z:
    z.write(pths[-1], 'libero.pth')
    for i in idx:
        if 'added' in os.path.basename(i) or len(idx) == 1:
            z.write(i, 'libero.index')
            break
print('modelo:', pths[-1], '| índice:', idx)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Voz Líbero', exist_ok=True)
    !cp /content/libero-rvc.zip "/content/drive/MyDrive/Voz Líbero/"
    print('Salvo no Drive: Voz Líbero/libero-rvc.zip')
except Exception as e:
    print('Drive não montado:', e)
from google.colab import files
files.download('/content/libero-rvc.zip')
